# Version 2 Notebook 01:
# Open-Meteo Single Runs Source Pilot

This notebook records the source audit supporting the
two-year weather-only training expansion.

The audit establishes the historical model cycles that can
be used consistently, verifies the reconstruction of the
Hong Kong local-day maximum temperature, and compares the
new extraction with the certified Version 1 forecasts.

No market price, realised market outcome, model fitting,
model selection or trading calculation is performed.

In [1]:
from pathlib import Path
import hashlib
import json

import pandas as pd


ROOT = Path.cwd().resolve()

for candidate in [ROOT, *ROOT.parents]:
    if (
        candidate
        / "config/v2/"
        "single_runs_pilot_spec.json"
    ).exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Repository root not found."
    )


def as_bool(series: pd.Series) -> pd.Series:
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .isin({"true", "1"})
    )


def sha256(path: Path) -> str:
    return hashlib.sha256(
        path.read_bytes()
    ).hexdigest()


MANIFEST = json.loads(
    (
        ROOT
        / "data/manifests/v2/"
        "01_single_runs_pilot_manifest.json"
    ).read_text(
        encoding="utf-8"
    )
)

SPEC = json.loads(
    (
        ROOT
        / "config/v2/"
        "single_runs_pilot_spec.json"
    ).read_text(
        encoding="utf-8"
    )
)

CHECKS = pd.read_csv(
    ROOT
    / "outputs/v2/diagnostics/"
    "01_single_runs_pilot_integrity_checks.csv"
)

PROBES = pd.read_csv(
    ROOT
    / "outputs/v2/diagnostics/"
    "01_single_runs_archive_probe_results.csv"
)

BOUNDARY = pd.read_csv(
    ROOT
    / "outputs/v2/diagnostics/"
    "01_single_runs_archive_boundary_audit.csv"
)

REQUEST_PLAN = pd.read_csv(
    ROOT
    / "outputs/v2/diagnostics/"
    "01_single_runs_pilot_request_plan.csv"
)

OVERLAP = pd.read_csv(
    ROOT
    / "outputs/v2/diagnostics/"
    "01_single_runs_overlap_reconstruction.csv"
)

SUMMARY = pd.read_csv(
    ROOT
    / "outputs/v2/diagnostics/"
    "01_single_runs_overlap_summary.csv"
)

In [2]:
print("=" * 76)
print("PHASE 2 SINGLE-RUNS SOURCE PILOT")
print("=" * 76)

print("Status:", MANIFEST["status"])
print("Phase status:", MANIFEST["phase_status"])

print(
    "Core historical cycles:",
    MANIFEST[
        "historical_training_core_cycles_utc"
    ],
)

print(
    "Supplementary cycles:",
    MANIFEST["supplementary_cycles_utc"],
)

print(
    "Core archive probes:",
    MANIFEST[
        "successful_archive_probe_runs"
    ],
    "/",
    MANIFEST["archive_probe_runs"],
)

print(
    "Overlap dates:",
    MANIFEST["overlap_dates"],
)

print(
    "Overlap rows:",
    MANIFEST["successful_overlap_rows"],
    "/",
    MANIFEST["overlap_rows"],
)

print(
    "Median absolute difference:",
    MANIFEST[
        "median_absolute_difference_c"
    ],
)

print(
    "Maximum absolute difference:",
    MANIFEST[
        "maximum_absolute_difference_c"
    ],
)

PHASE 2 SINGLE-RUNS SOURCE PILOT
Status: SINGLE_RUNS_PILOT_APPROVED
Phase status: PHASE2_COMPLETE
Core historical cycles: [0, 12]
Supplementary cycles: [6, 18]
Core archive probes: 4 / 4
Overlap dates: 5
Overlap rows: 20 / 20
Median absolute difference: 0.0
Maximum absolute difference: 0.0


In [3]:
print("Required integrity checks:")

required = CHECKS.loc[
    as_bool(CHECKS["required"])
]

print(
    required.to_string(
        index=False
    )
)

print()
print("Archive probes:")

print(
    PROBES[
        [
            "run_init_utc",
            "request_succeeded",
            "hourly_rows",
            "returned_timezone",
        ]
    ].to_string(
        index=False
    )
)

print()
print("Overlap summary:")

print(
    SUMMARY.to_string(
        index=False
    )
)

Required integrity checks:
                           check  required  passed         detail
          archive_probe_requests      True    True            4/4
                overlap_requests      True    True          20/20
target_dates_have_24_local_hours      True    True          20/20
 runs_available_before_decisions      True    True          20/20
               returned_timezone      True    True Asia/Hong_Kong

Archive probes:
    run_init_utc  request_succeeded  hourly_rows returned_timezone
2024-03-14T00:00               True          240    Asia/Hong_Kong
2024-03-14T12:00               True          240    Asia/Hong_Kong
2025-03-15T00:00               True          240    Asia/Hong_Kong
2025-03-15T12:00               True          240    Asia/Hong_Kong

Overlap summary:
 overlap_dates  overlap_rows  successful_rows  complete_24_hour_rows  availability_pass_rows  mean_absolute_difference_c  median_absolute_difference_c  maximum_absolute_difference_c
             5           

In [4]:
archive_start = BOUNDARY.loc[
    BOUNDARY["window"].eq(
        "archive_start"
    )
].copy()

archive_start["usable"] = (
    as_bool(
        archive_start[
            "request_succeeded"
        ]
    )
    & pd.to_numeric(
        archive_start["hourly_rows"],
        errors="coerce",
    ).ge(24)
    & archive_start[
        "returned_timezone"
    ].eq("Asia/Hong_Kong")
)

cycle_coverage = (
    archive_start.groupby(
        "cycle_utc",
        as_index=False,
    )
    .agg(
        tested_dates=(
            "run_date",
            "nunique",
        ),
        successful_dates=(
            "usable",
            "sum",
        ),
    )
)

print(
    "Archive-start cycle coverage:"
)

print(
    cycle_coverage.to_string(
        index=False
    )
)

Archive-start cycle coverage:
 cycle_utc  tested_dates  successful_dates
         0            18                18
         6            18                 0
        12            18                18
        18            18                 0


In [5]:
acceptable_statuses = {
    "SINGLE_RUNS_PILOT_APPROVED",
    (
        "SINGLE_RUNS_PILOT_APPROVED_"
        "WITH_OVERLAP_DIFFERENCES"
    ),
}

assert MANIFEST["status"] in acceptable_statuses
assert MANIFEST["phase_status"] == "PHASE2_COMPLETE"
assert MANIFEST["pilot_approved"]

assert (
    MANIFEST[
        "historical_training_core_cycles_utc"
    ]
    == [0, 12]
)

assert (
    MANIFEST["supplementary_cycles_utc"]
    == [6, 18]
)

assert not MANIFEST[
    "four_cycle_archive_completeness_assumed"
]

assert (
    MANIFEST[
        "successful_archive_probe_runs"
    ]
    == 4
)

assert MANIFEST["archive_probe_runs"] == 4
assert MANIFEST["successful_overlap_rows"] == 20
assert MANIFEST["overlap_rows"] == 20
assert MANIFEST["complete_24_hour_rows"] == 20

assert as_bool(
    required["passed"]
).all()

assert len(REQUEST_PLAN) == 20
assert len(OVERLAP) == 20
assert OVERLAP["target_date"].nunique() == 5
assert OVERLAP["decision_rule"].nunique() == 4
assert OVERLAP["local_hour_count"].eq(24).all()

assert as_bool(
    OVERLAP["available_before_decision"]
).all()

assert (
    pd.to_numeric(
        OVERLAP[
            "absolute_difference_c"
        ],
        errors="coerce",
    )
    .fillna(float("inf"))
    .max()
    == 0.0
)

for relative_path, expected_hash in (
    MANIFEST[
        "tracked_evidence_hashes"
    ].items()
):
    actual_hash = sha256(
        ROOT / relative_path
    )

    assert actual_hash == expected_hash

assert not MANIFEST["market_prices_accessed"]
assert not MANIFEST["realised_outcomes_accessed"]
assert not MANIFEST["model_fitted"]
assert not MANIFEST["model_selected"]
assert not MANIFEST["version_1_modified"]

print()
print(
    "NOTEBOOK 01 SOURCE PILOT: PASSED"
)

print(
    "Next stage: construct the complete "
    "two-year weather-only request plan."
)


NOTEBOOK 01 SOURCE PILOT: PASSED
Next stage: construct the complete two-year weather-only request plan.
